# 2 · Bronze, from a database

In notebook 1 you learned what a pipeline is. Now you write one, **in these
cells**, one component at a time, and watch each part produce output before the
next part uses it.

At the end you run the packaged version and confirm it does exactly the same
thing.

| | |
|---|---|
| **reads** | `kerb.trips` in PostgreSQL, somebody else's live table |
| **writes** | `teach.bronze_trips` in PostgreSQL, our copy |
| **runs** | hourly |

![](img/bronze-1-copy.png)

In [ ]:
import sys; sys.path.insert(0, '.')
from nb import show, sql, fetch, run, counts      # formatting helpers, nothing clever

import datetime as dt
import psycopg
from pipelines.lib.config import dsn, SCHEMA      # the one file that knows the addresses

print('our schema:', SCHEMA)

---

## Step 0 · Look at the source before you touch it

Nobody writes a pipeline against a table they have not looked at. Two questions
first: what does a row look like, and how much of it is there.

In [ ]:
sql("""
    SELECT trip_id, requested_at, rider_id, driver_id, pu_zone_id,
           distance_km, duration_s, status
    FROM kerb.trips
    ORDER BY requested_at DESC
    LIMIT 5
""", 'kerb.trips, the newest five rides')

In [ ]:
sql("""
    SELECT count(*)                    AS rides,
           min(date(requested_at))     AS first_day,
           max(date(requested_at))     AS last_day,
           count(DISTINCT status)      AS distinct_statuses
    FROM kerb.trips
""", 'how much is there')

Three hundred and sixty thousand rides across a month. **A pipeline never reads
all of that.** In six months this table has fifty million rows and copying them
hourly is absurd.

So we rebuild a small window of recent days, over and over.

---

## Step 1 · Choose the window, and get it right

![](img/bronze-2-window.png)

The obvious answer is *"the last seven days, counting back from today"*, and it
is wrong here, twice over. Run this and see why.

In [ ]:
with psycopg.connect(dsn()) as c:
    today_style = c.execute("""
        SELECT count(*) FROM kerb.trips
        WHERE requested_at >= current_date - interval '7 days'
    """).fetchone()[0]

    max_day = c.execute('SELECT max(date(requested_at)) FROM kerb.trips').fetchone()[0]

    rides_on_max = c.execute(
        'SELECT count(*) FROM kerb.trips WHERE date(requested_at) = %s', (max_day,)).fetchone()[0]

print(f'counting back from today   : {today_style:,} rides')
print(f'max(date) in the table     : {max_day}, which holds {rides_on_max:,} rides')

**Zero.** The dataset was generated once and then sat still, so nothing is near
today, and you would spend twenty minutes wondering why your pipeline reads
nothing.

And `max(date)` is only safe until one stray test row dated next year drags the
anchor to a day holding a single ride.

So anchor on **the newest day that actually looks like a day of trading**.

In [ ]:
def choose_window(days):
    """Return (lo, hi) for the last `days` real days of trading."""
    with psycopg.connect(dsn()) as c:
        newest = c.execute("""
            SELECT date(requested_at)
            FROM kerb.trips
            GROUP BY 1
            HAVING count(*) > 100        -- ignore days with a handful of stray rows
            ORDER BY 1 DESC
            LIMIT 1
        """).fetchone()[0]

    # lo is inclusive, hi is exclusive. Always. Mixing those up is how you get a
    # pipeline that double counts one day, every single run.
    lo = newest - dt.timedelta(days=days - 1)
    hi = newest + dt.timedelta(days=1)
    return lo, hi

lo, hi = choose_window(7)
print(f'rebuilding {lo}  <=  requested_at  <  {hi}')

### Say the two bounds out loud

> **lo is inclusive. hi is exclusive.**

`hi` is the day *after* the last day we want. That is why the query below uses
`>= lo` and `< hi`, never `BETWEEN`. `BETWEEN` is inclusive at both ends, and a
pipeline that double counts one day every run is almost impossible to spot,
because every total looks *almost* right.

---

## Step 2 · Read that window out of the source

One query, one window. **No JOIN and no aggregation.** Bronze copies one source
table and nothing else. If you find yourself joining in a bronze pipeline, you
are building silver and should say so.

In [ ]:
READ = """
    SELECT trip_id,
           date(requested_at) AS trip_date,
           rider_id,
           driver_id,
           pu_zone_id,
           distance_km,
           duration_s,
           status
    FROM kerb.trips
    WHERE requested_at >= %s        -- inclusive
      AND requested_at <  %s        -- exclusive
    ORDER BY requested_at
"""

with psycopg.connect(dsn()) as c:
    rows = c.execute(READ, (lo, hi)).fetchall()

print(f'{len(rows):,} rides in the window')
print()
for r in rows[:3]:
    print(' ', r)

`rows` is now a list of plain python tuples. **That is all a pipeline ever
holds**: a batch of records it read, on the way to somewhere else.

---

## Step 3 · Make somewhere to put them

The table is created by the pipeline, not by a human running SQL by hand at some
point in the past that nobody wrote down. `IF NOT EXISTS` means a brand new
laptop and a three year old production database end up with exactly the same
shape.

In [ ]:
DDL = f"""
CREATE TABLE IF NOT EXISTS {SCHEMA}.bronze_trips (
    trip_id      TEXT PRIMARY KEY,   -- one row per ride, enforced by the database
    trip_date    DATE NOT NULL,      -- derived from requested_at, so we can rebuild a window
    rider_id     TEXT,
    driver_id    TEXT,               -- empty until a driver accepts, so nullable
    pickup_zone  INT,
    distance_km  NUMERIC(8,2),
    duration_s   INT,                -- seconds, exactly as the source has it
    status       TEXT NOT NULL
);
CREATE INDEX IF NOT EXISTS ix_bronze_trips_date ON {SCHEMA}.bronze_trips (trip_date);
"""

with psycopg.connect(dsn(), autocommit=True) as c:
    c.execute(DDL)

sql(f"""
    SELECT column_name, data_type, is_nullable
    FROM information_schema.columns
    WHERE table_schema = '{SCHEMA}' AND table_name = 'bronze_trips'
    ORDER BY ordinal_position
""", 'the table we just created')

### Every column maps to one column in the source

Nothing invented, nothing renamed for taste. `trip_date` is the only derived
column, and it exists only so we have something to partition the window on in
step 5.

---

## Step 4 · The contract

A contract is **the set of values we already know how to interpret**. It is not
a validation rule and it is not a preference. It is a written record of what the
rest of the company has agreed a ride can be.

In [ ]:
KNOWN_STATUS = {'completed', 'cancelled_rider', 'cancelled_driver', 'no_driver'}

sql("""
    SELECT status, count(*) AS rides
    FROM kerb.trips
    GROUP BY 1 ORDER BY 2 DESC
""", 'what the source actually sends today')

Four statuses, and our contract knows all four. **If a fifth ever appears**, one
of two things has happened: the product team shipped a new state and forgot to
tell anybody, or something upstream is corrupting the field.

Both are things you want to hear about on the day, not in three weeks.

![](img/bronze-3-piles.png)

## Now sort the batch into two piles

To see the held pile do its job, we pretend the source sent us one ride with a
status nobody has ever agreed on.

In [ ]:
batch = list(rows)
batch.append(('TRP-PRETEND-1', lo, 'RDR000001', 'DRV000001', 42, 3.4, 600,
              'refunded_by_support'))       # <- a status nobody agreed on

keep, held = [], []
for rec in batch:
    status = rec[7]
    if status not in KNOWN_STATUS:
        held.append((rec, f'status {status!r} is not one of {sorted(KNOWN_STATUS)}'))
    else:
        keep.append(rec)

print(f'landed : {len(keep):,}')
print(f'held   : {len(held):,}')
for rec, reason in held:
    print(f'\n  {rec[0]}')
    print(f'  {reason}')

### Held, not dropped, and not defaulted

The held record keeps **the record, the reason, and the original values**, so
the person who picks it up tomorrow has everything they need and does not have
to guess.

In the packaged pipeline this is `run.quarantine(...)`, which writes to a real
`teach.quarantine` table. You will read that table in notebook 9.

---

## Step 5 · Write the window, all at once or not at all

This is what makes the pipeline safe to run twice, and it is four lines.

Instead of appending, we **delete the window we are about to rebuild** and
insert the new rows **inside the same transaction**.

In [ ]:
INSERT = f"""
    INSERT INTO {SCHEMA}.bronze_trips
        (trip_id, trip_date, rider_id, driver_id, pickup_zone, distance_km, duration_s, status)
    VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
"""

def write_window(records, lo, hi):
    # autocommit=False: nothing is visible to anyone else until we commit
    with psycopg.connect(dsn(), autocommit=False) as c, c.cursor() as cur:
        cur.execute(f'DELETE FROM {SCHEMA}.bronze_trips '
                    f'WHERE trip_date >= %s AND trip_date < %s', (lo, hi))
        deleted = cur.rowcount
        cur.executemany(INSERT, records)
        c.commit()
    return deleted

deleted = write_window(keep, lo, hi)
print(f'deleted {deleted:,} rows, then inserted {len(keep):,}')

after = fetch(f'SELECT count(*) AS n FROM {SCHEMA}.bronze_trips').n[0]
print(f'the table now holds {after:,} rows')

## Run the exact same cell again

This is the property worth proving in front of the room.

In [ ]:
deleted = write_window(keep, lo, hi)
again = fetch(f'SELECT count(*) AS n FROM {SCHEMA}.bronze_trips').n[0]

print(f'deleted {deleted:,}, inserted {len(keep):,}')
print(f'the table now holds {again:,} rows')
print()
print('same number' if again == after else 'DIFFERENT, something is wrong')

### Two properties fell out of those four lines

**Run it twice and you get the same table, not double the rows.** The delete
removes what the previous run wrote before the insert puts it back. That is
called being **idempotent**, and without it nobody can ever safely rerun
anything.

**Kill it halfway and nothing is lost.** The delete is rolled back along with
the insert, so a reader never sees a table with a hole in it. There is no moment
where the data is half deleted.

---

## And now the packaged pipeline

Everything above, in one file, plus the run log and the real quarantine table.

In [ ]:
run('-m', 'pipelines.p1_bronze_trips', '--days', '7')

In [ ]:
sql(f"""
    SELECT pipeline, status, rows_in, rows_out,
           round(extract(epoch from (ended_at - started_at))::numeric, 2) AS secs, message
    FROM {SCHEMA}.runs
    WHERE pipeline = 'p1_bronze_trips'
    ORDER BY started_at DESC LIMIT 3
""", 'the run log, which every pipeline writes')

Note `rows_in` and `rows_out` are the **same number**. Nothing was held, because
the real source only ever sends the four statuses we agreed on. The one held
record earlier was our own invention.

## What is in the warehouse now

In [ ]:
counts()

---

## What you learned

- **Look at the source first.** Shape, size, and date range, before any code
- A pipeline rebuilds **a window**, never the whole table
- Anchor the window **on the data**, not on today, and not on `max(date)`
- **lo inclusive, hi exclusive**, and never `BETWEEN`
- Bronze **copies**. No joins, no renames, no cleaning
- A contract is what you already know how to read. An unknown value is **held**,
  not dropped and not defaulted
- **Delete the window, then insert, in one transaction.** That one habit is what
  makes a pipeline safe to rerun